# 🧪 UMKM Copywriting Generator — Notebook Eksperimen

Notebook ini digunakan untuk:
1. Eksplorasi knowledge base
2. Uji coba retrieval FAISS
3. Uji coba end-to-end RAG pipeline
4. Evaluasi ROUGE dan response time
5. Visualisasi hasil eksperimen top-k

---
> **Pastikan** kamu sudah menjalankan `python embedder.py` di root project sebelum menjalankan notebook ini.

In [ ]:
# Setup path agar bisa import module dari root project
import sys
import os

# Tambahkan root project ke sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

os.chdir(ROOT)  # Set working directory ke root agar path relatif (data/, index/) bekerja
print(f"Working directory: {os.getcwd()}")

## 1. Eksplorasi Knowledge Base

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load knowledge base
with open("data/knowledge_base.json", "r", encoding="utf-8") as f:
    kb = json.load(f)

df_kb = pd.DataFrame(kb)
print(f"Total entri: {len(df_kb)}")
df_kb.head()

In [ ]:
# Distribusi per kategori
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
cat_counts = df_kb["kategori"].value_counts()
axes[0].bar(cat_counts.index, cat_counts.values, color=sns.color_palette("Set2"))
axes[0].set_title("Distribusi Entri per Kategori", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Kategori")
axes[0].set_ylabel("Jumlah")
for i, v in enumerate(cat_counts.values):
    axes[0].text(i, v + 0.2, str(v), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(
    cat_counts.values,
    labels=cat_counts.index,
    autopct="%1.0f%%",
    colors=sns.color_palette("Set2"),
    startangle=90,
)
axes[1].set_title("Proporsi Kategori", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("outputs/distribusi_kategori.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nDistribusi:")
print(cat_counts)

In [ ]:
# Statistik panjang teks
df_kb["len_deskripsi"] = df_kb["deskripsi_produk"].str.len()
df_kb["len_caption"] = df_kb["caption_ig"].str.len()
df_kb["len_tagline"] = df_kb["tagline"].str.len()
df_kb["len_keunggulan"] = df_kb["keunggulan"].str.len()

print("Statistik panjang teks (karakter):")
df_kb[["len_deskripsi", "len_caption", "len_tagline", "len_keunggulan"]].describe().round(1)

In [ ]:
# Distribusi panjang deskripsi per kategori
fig, ax = plt.subplots(figsize=(10, 5))
for cat in df_kb["kategori"].unique():
    subset = df_kb[df_kb["kategori"] == cat]["len_deskripsi"]
    ax.hist(subset, alpha=0.6, label=cat, bins=10)

ax.set_title("Distribusi Panjang Deskripsi per Kategori", fontsize=13, fontweight="bold")
ax.set_xlabel("Panjang (karakter)")
ax.set_ylabel("Frekuensi")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/distribusi_panjang_deskripsi.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Uji Coba Retrieval FAISS

In [ ]:
from embedder import UMKMEmbedder

embedder = UMKMEmbedder()
embedder.load_index()

In [ ]:
# Test retrieval: query makanan
query_makanan = "keripik singkong pedas manis renyah camilan"
results = embedder.retrieve(query_makanan, k=5)

print(f"Query: '{query_makanan}'\n")
print(f"{'Rank':<6} {'Score':<8} {'Kategori':<12} {'Nama Produk'}")
print("-" * 60)
for i, r in enumerate(results, 1):
    d = r["document"]
    print(f"{i:<6} {r['score']:.4f}   {d['kategori']:<12} {d['nama_produk']}")

In [ ]:
# Test retrieval: query minuman
query_minuman = "kopi arabika single origin fruity aroma"
results_minuman = embedder.retrieve(query_minuman, k=5)

print(f"Query: '{query_minuman}'\n")
print(f"{'Rank':<6} {'Score':<8} {'Kategori':<12} {'Nama Produk'}")
print("-" * 60)
for i, r in enumerate(results_minuman, 1):
    d = r["document"]
    print(f"{i:<6} {r['score']:.4f}   {d['kategori']:<12} {d['nama_produk']}")

In [ ]:
# Test retrieval: query kosmetik
query_kosmetik = "serum wajah vitamin C mencerahkan BPOM"
results_kosmetik = embedder.retrieve(query_kosmetik, k=5)

print(f"Query: '{query_kosmetik}'\n")
print(f"{'Rank':<6} {'Score':<8} {'Kategori':<12} {'Nama Produk'}")
print("-" * 60)
for i, r in enumerate(results_kosmetik, 1):
    d = r["document"]
    print(f"{i:<6} {r['score']:.4f}   {d['kategori']:<12} {d['nama_produk']}")

In [ ]:
# Visualisasi retrieval score per query
queries_test = [
    ("Makanan", query_makanan, results),
    ("Minuman", query_minuman, results_minuman),
    ("Kosmetik", query_kosmetik, results_kosmetik),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (label, query, res) in zip(axes, queries_test):
    names = [r["document"]["nama_produk"][:25] for r in res]
    scores = [r["score"] for r in res]
    bars = ax.barh(range(len(names)), scores, color=sns.color_palette("Blues_r", len(names)))
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlim(0, 1)
    ax.set_title(f"Top-5 Retrieval: {label}", fontweight="bold")
    ax.set_xlabel("Cosine Similarity")
    for bar, score in zip(bars, scores):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f"{score:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("outputs/retrieval_scores.png", dpi=150, bbox_inches="tight")
plt.show()

## 3. Uji Coba End-to-End RAG Pipeline

In [ ]:
from rag_pipeline import RAGPipeline

pipeline = RAGPipeline()
print("Pipeline siap!")

In [ ]:
import time

# Test generate untuk produk makanan
start = time.time()
result = pipeline.generate(
    nama_produk="Keripik Tempe Bawang Renyah",
    kategori="makanan",
    keunggulan="tempe segar, bumbu bawang putih, renyah tahan lama, tanpa pengawet",
    k=3
)
elapsed = time.time() - start

print(f"Response time: {elapsed:.2f}s\n")
print("=" * 60)
print("DESKRIPSI PRODUK:")
print(result["deskripsi"])
print("\nCAPTION INSTAGRAM:")
print(result["caption_ig"])
print("\nTAGLINE:")
print(result["tagline"])
print("=" * 60)
print(f"\nRetrieved {len(result['retrieved_docs'])} dokumen referensi:")
for i, doc in enumerate(result["retrieved_docs"], 1):
    print(f"  {i}. {doc['document']['nama_produk']} (score: {doc['score']:.4f})")

In [ ]:
# Test generate untuk produk kosmetik
start = time.time()
result_kosmetik = pipeline.generate(
    nama_produk="Body Lotion Aloe Vera Collagen",
    kategori="kosmetik",
    keunggulan="aloe vera organik, collagen booster, kulit lembab 24 jam, BPOM terdaftar",
    k=3
)
elapsed = time.time() - start

print(f"Response time: {elapsed:.2f}s\n")
print("=" * 60)
print("DESKRIPSI PRODUK:")
print(result_kosmetik["deskripsi"])
print("\nCAPTION INSTAGRAM:")
print(result_kosmetik["caption_ig"])
print("\nTAGLINE:")
print(result_kosmetik["tagline"])
print("=" * 60)

In [ ]:
# Test generate untuk produk kerajinan
start = time.time()
result_kerajinan = pipeline.generate(
    nama_produk="Anyaman Tas Bambu Eco",
    kategori="kerajinan",
    keunggulan="bambu lokal, anyam tangan, eco-friendly, desain minimalis",
    k=3
)
elapsed = time.time() - start

print(f"Response time: {elapsed:.2f}s\n")
print("=" * 60)
print("DESKRIPSI PRODUK:")
print(result_kerajinan["deskripsi"])
print("\nCAPTION INSTAGRAM:")
print(result_kerajinan["caption_ig"])
print("\nTAGLINE:")
print(result_kerajinan["tagline"])
print("=" * 60)

## 4. Evaluasi ROUGE & Response Time

In [ ]:
from evaluator import evaluate

# Jalankan evaluasi penuh (20 test cases, k=3)
df_eval = evaluate(top_k=3, save_csv=True)

In [ ]:
# Tampilkan hasil evaluasi
display_cols = ["nama_produk", "kategori", "rouge1_f", "rouge2_f", "rougeL_f",
                "top_retrieval_score", "response_time_s"]
df_eval[display_cols].style.background_gradient(subset=["rouge1_f", "rougeL_f"], cmap="YlGn")

In [ ]:
# Visualisasi ROUGE score per kategori
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ["rouge1_f", "rouge2_f", "rougeL_f"]
titles = ["ROUGE-1", "ROUGE-2", "ROUGE-L"]
colors = ["#4CAF50", "#2196F3", "#FF9800"]

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    per_cat = df_eval.groupby("kategori")[metric].mean().sort_values(ascending=False)
    bars = ax.bar(per_cat.index, per_cat.values, color=color, alpha=0.8)
    ax.set_title(f"{title} per Kategori", fontweight="bold")
    ax.set_ylabel("F-measure")
    ax.set_ylim(0, 1)
    ax.tick_params(axis="x", rotation=15)
    for bar, val in zip(bars, per_cat.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", fontsize=9, fontweight="bold")

plt.suptitle("ROUGE Score per Kategori (k=3)", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("outputs/rouge_per_kategori.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Visualisasi response time
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Response time per produk
axes[0].barh(
    df_eval["nama_produk"],
    df_eval["response_time_s"],
    color=sns.color_palette("coolwarm", len(df_eval))
)
axes[0].axvline(df_eval["response_time_s"].mean(), color="red", linestyle="--",
                label=f'Mean: {df_eval["response_time_s"].mean():.2f}s')
axes[0].set_title("Response Time per Produk", fontweight="bold")
axes[0].set_xlabel("Waktu (detik)")
axes[0].legend()
axes[0].tick_params(axis="y", labelsize=8)

# Scatter: retrieval similarity vs ROUGE-L
for cat in df_eval["kategori"].unique():
    subset = df_eval[df_eval["kategori"] == cat]
    axes[1].scatter(subset["top_retrieval_score"], subset["rougeL_f"], label=cat, s=80, alpha=0.8)
axes[1].set_title("Retrieval Similarity vs ROUGE-L", fontweight="bold")
axes[1].set_xlabel("Top Retrieval Score (cosine)")
axes[1].set_ylabel("ROUGE-L F-measure")
axes[1].legend()

plt.tight_layout()
plt.savefig("outputs/response_time_dan_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Eksperimen Pengaruh Top-k

In [ ]:
from evaluator import evaluate_topk_experiment

# Jalankan eksperimen top-k (k=1,3,5)
df_topk = evaluate_topk_experiment(k_values=[1, 3, 5])
df_topk

In [ ]:
# Visualisasi eksperimen top-k
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

k_labels = [str(k) for k in df_topk["top_k"]]

# ROUGE-1
axes[0].bar(k_labels, df_topk["rouge1_avg"], color="#4CAF50", alpha=0.85)
axes[0].set_title("ROUGE-1 avg vs top-k", fontweight="bold")
axes[0].set_xlabel("top-k")
axes[0].set_ylabel("ROUGE-1 avg")
axes[0].set_ylim(0, max(df_topk["rouge1_avg"]) * 1.3)
for i, v in enumerate(df_topk["rouge1_avg"]):
    axes[0].text(i, v + 0.005, f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")

# ROUGE-L
axes[1].bar(k_labels, df_topk["rougeL_avg"], color="#FF9800", alpha=0.85)
axes[1].set_title("ROUGE-L avg vs top-k", fontweight="bold")
axes[1].set_xlabel("top-k")
axes[1].set_ylabel("ROUGE-L avg")
axes[1].set_ylim(0, max(df_topk["rougeL_avg"]) * 1.3)
for i, v in enumerate(df_topk["rougeL_avg"]):
    axes[1].text(i, v + 0.005, f"{v:.4f}", ha="center", fontsize=10, fontweight="bold")

# Response Time
axes[2].bar(k_labels, df_topk["response_time_avg_s"], color="#9C27B0", alpha=0.85)
axes[2].set_title("Response Time avg vs top-k", fontweight="bold")
axes[2].set_xlabel("top-k")
axes[2].set_ylabel("Waktu (detik)")
axes[2].set_ylim(0, max(df_topk["response_time_avg_s"]) * 1.3)
for i, v in enumerate(df_topk["response_time_avg_s"]):
    axes[2].text(i, v + 0.05, f"{v:.2f}s", ha="center", fontsize=10, fontweight="bold")

plt.suptitle("Eksperimen Pengaruh Top-k terhadap Kualitas Output",
             fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("outputs/topk_experiment_viz.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nTabel hasil eksperimen top-k:")
print(df_topk.to_string(index=False))

## 6. Analisis Raw Output LLM

In [ ]:
# Lihat raw output LLM (sebelum parsing)
start = time.time()
result_raw = pipeline.generate(
    nama_produk="Batik Tulis Solo Premium",
    kategori="fashion",
    keunggulan="batik tulis tangan, pewarna alami, motif klasik Solo, sertifikat keaslian",
    k=3
)
elapsed = time.time() - start

print(f"Response time: {elapsed:.2f}s\n")
print("=" * 60)
print("RAW OUTPUT dari LLM:")
print("=" * 60)
print(result_raw["raw_output"])
print("=" * 60)
print("\nHASIL PARSING:")
print(f"Deskripsi : {result_raw['deskripsi'][:80]}...")
print(f"Caption   : {result_raw['caption_ig'][:80]}...")
print(f"Tagline   : {result_raw['tagline']}")

In [ ]:
# Ringkasan akhir semua hasil
if 'df_eval' in dir():
    print("=" * 60)
    print("RINGKASAN EVALUASI AKHIR")
    print("=" * 60)
    print(f"Total test cases   : {len(df_eval)}")
    print(f"ROUGE-1 avg        : {df_eval['rouge1_f'].mean():.4f}")
    print(f"ROUGE-2 avg        : {df_eval['rouge2_f'].mean():.4f}")
    print(f"ROUGE-L avg        : {df_eval['rougeL_f'].mean():.4f}")
    print(f"Retrieval sim avg  : {df_eval['top_retrieval_score'].mean():.4f}")
    print(f"Response time avg  : {df_eval['response_time_s'].mean():.2f}s")
    print()
    print("ROUGE-L per kategori:")
    print(df_eval.groupby("kategori")["rougeL_f"].mean().sort_values(ascending=False).to_string())
    print("=" * 60)